In [ ]:
import os
os.chdir('???')
os.getcwd()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Load dataset

In [ ]:
orig_df = pd.read_csv("bike_day.csv")
orig_df.head(10)

In [ ]:
df = orig_df.drop(["instant", "dteday"], axis=1)
df = df.copy()
orig_X = df.drop("cnt", axis=1)
X = orig_X.copy()

## Handling Missing Values

In [ ]:
# Plot heatmap to visualize locations of missing values
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')

In [ ]:
# find missing value percent for each variable
null_percent = df.isnull().sum()/len(df)*100
null_percent

## Handling Outliers

In [ ]:
def remove_outliers_iqr(df, outlier_vars):
    temp_df = df.copy()
    for _var in outlier_vars:
        q1 = temp_df[_var].quantile(0.25)
        q3 = temp_df[_var].quantile(0.75)
        iqr = q3-q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        mask = (temp_df[_var] < lower_bound) | (temp_df[_var] > upper_bound)
        temp_df = temp_df[~mask]
        print(temp_df.shape)
    return temp_df

## Handling Skewness

In [ ]:
X.hist()

In [ ]:
def transform_log(X, skew_var):
    temp_X = X.copy()
    temp_X[skew_var] = np.log1p(temp_X[skew_var])   # log1p = log(1 + y)
    return temp_X
    
def transform_boxcox(X, skew_var):
    temp_X = X.copy()
    from scipy.stats import boxcox
    temp_bc, lambda_bc = boxcox(temp_X[skew_var])
    temp_X[skew_var] = temp_bc
    return temp_X

def transform_yeojohnson(X, skew_var):
    temp_X = X.copy()
    from sklearn.preprocessing import PowerTransformer
    pt = PowerTransformer(method='yeo-johnson')
    temp_X[skew_var] = pt.fit_transform(temp_X[[skew_var]])
    return temp_X

In [ ]:
## Example
'''
skew_var = 'windspeed'
X_train = transform_log(X_train, skew_var)
'''

## Scaling

In [ ]:
# Scale by standardized normal distribution, (x-mean)/sd
def scale_standard(X_train, X_test):
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

# Scale by min-max, (x-min)/(max-min)
def scale_minmax(X_train, X_test):
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

# Scale by robust, (x-median)/iqr
def scale_minmax(X_train, X_test):
    from sklearn.preprocessing import RobustScaler
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

In [ ]:
### Example 
'''
X_train, X_test = scale_standard(X_train, X_test)
'''

## Checking Multicollinearity of X

In [ ]:
# Plot heatmap of correlation coefficients 
sns.heatmap(df.corr(),
            square=True,
            linewidths=0.25,    
            linecolor=(0,0,0),
            cmap=sns.color_palette("coolwarm"),
            annot=True)
plt.show()

# | correlation |< 0.5: Low to moderate correlation.  Usually not a multicollinearity concern 
# | correlation | ≥ 0.7: high correlation.  Possible multicollinearity

In [ ]:
def check_VIF(X):
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    temp_df = X.assign(const=1)
    vif = pd.DataFrame()
    vif["feature"] = temp_df.columns
    vif["VIF"] = [variance_inflation_factor(temp_df.values, i) for i in range(temp_df.shape[1])]
    
    # Remove intercept row
    vif_df = vif[vif["feature"] != "const"].reset_index(drop=True)
    return vif_df
    
    #VIF > 5 → high multicollinearity (common rule of thumb)
    #VIF > 10 → serious multicollinearity
    #VIF = 1 → no multicollinearity
    #VIF is based on R² when regressing one feature on all other feature

In [ ]:
check_VIF(X)

## Performance Evaluation

In [ ]:
def compute_regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

def compute_mape(y_true, y_pred):
    # Mean Absolute Percentage Error
    # Note: MAPE fails if y_true contains zeros.
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def compute_smape(y_true, y_pred):
    # Symmetric Mean Absolute Percentage Error
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

def compute_rmsle(y_true, y_pred, neg_mask=0):
    # Root Mean Squared Log Error
    # RMSLE is good for right-skewed data
    # Note: log1p(x) = log(1+x), and the log parameter must be positive. 
    # Therefore, x >= -1. Otherwise, log1p(1+x) encounters an invalid value.
        
    if neg_mask == 1:
        mask = (y_true >= 0) & (y_pred >= 0)
        y_true = y_true[mask]
        y_pred = y_pred[mask]
    
    np_y_true, np_y_pred = np.array(y_true), np.array(y_pred)
    rmsle = np.sqrt(np.mean((np.log1p(np_y_true) - np.log1p(np_y_pred))**2))  # Root Mean Squared Log Error

    return rmsle

## Handling Residuals

In [ ]:
# Residuals vs Fitted Plot
def create_residuals_fitted_plot(y_true, y_pred):
    residuals = y_true - y_pred
    
    plt.figure(figsize=(7, 5))
    plt.scatter(y_pred, residuals)
    plt.axhline(y=0, color='red', linestyle='--')
    plt.xlabel("Fitted Values (Predicted)")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Fitted Plot")
    plt.show()

In [ ]:
# Residual qqplot
def create_residuals_qqplot(y_true, y_pred):
    residuals = y_true - y_pred
    sm.qqplot(residuals, line='45', fit=True)
    plt.title("Q-Q Plot - Statsmodels")
    plt.show()

## Check Extreme Values

In [ ]:
def check_extreme_values(X_true, y_true, y_pred):
    residuals = y_true - y_pred
    
    X_computed = np.hstack([np.ones((X_true.shape[0], 1)), X_true])  # Add intercept column
    n, p = X_true.shape
    
    mse = np.sum(residuals**2) / (n - p)
    
    XtX_inv = np.linalg.inv(X_computed.T @ X_computed)  # @ = matrix multiplication operator.
    hat_matrix = X_computed @ XtX_inv @ X_computed.T 
    
    # Leverage = Outliers in X-direction = Extreme predictor values
    leverage = np.diag(hat_matrix)     
    leverage_threshold = 2 * p / n  # if leverage > 2p/n or 3p/n, leverage is considered high
    print("Leverage thresold:", leverage_threshold)
    
    # Leverage = Outliers in Y-direction = Extreme response values
    standardized_residuals = residuals / np.sqrt(mse * (1 - leverage))
    print("Standardized residuals thresold:", 2)
    
    cooks_distance = (standardized_residuals**2 / p) * (leverage / (1 - leverage))
    cooks_threshold = 4 / n
    print("Cooks thresold:", cooks_threshold)    
    cooks_df = cooks_distance[cooks_distance > cooks_threshold]
    index_to_remove = cooks_df.index.tolist()
    
    return leverage, leverage_threshold, standardized_residuals, cooks_distance, cooks_threshold, index_to_remove

In [ ]:
def create_cooks_plot(X_true, cooks_distance, cooks_threshold):
    n, p = X_true.shape
    plt.figure(figsize=(10, 5))
    plt.bar(np.arange(n), cooks_distance, color='skyblue', edgecolor='k')
    plt.xlabel("Observation Index")
    plt.ylabel("Cook's Distance")
    plt.title("Cook's Distance Bar Plot")
    
    # Threshold line for influential points
    plt.axhline(y=cooks_threshold, color='red', linestyle='--', label=f"Threshold = {cooks_threshold:.4f}")
    plt.legend()
    plt.show()

In [ ]:
## Example of checking extreme values for the first time
'''
leverage, leverage_threshold, standardized_residuals, cooks_distance, cooks_threshold, index_to_remove = check_extreme_values(X_train, y_train, y_train_pred)
main_index_to_remove = index_to_remove
print(main_index_to_remove)
print(len(main_index_to_remove)/df.shape[0])
create_cooks_plot(X_train, cooks_distance, cooks_threshold)
'''

In [ ]:
## Example of checking extreme values after the first time
'''
leverage, leverage_threshold, standardized_residuals, cooks_distance, cooks_threshold, index_to_remove = check_extreme_values(X_train, y_train, y_train_pred)
past_main_index_to_remove = main_index_to_remove
main_index_to_remove = main_index_to_remove + index_to_remove
print(main_index_to_remove)
print(len(main_index_to_remove)/df.shape[0])
create_cooks_plot(X_train, cooks_distance, cooks_threshold)
'''

## Check Feature Importance

In [ ]:
def get_permutation_importance(model, X, X_true, y_true):
    from sklearn.inspection import permutation_importance
    result = permutation_importance(model, X_test, y_test, n_repeats=10)
    
    perm_df = pd.DataFrame({
        "feature": X.columns,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std
    }).sort_values("importance_mean", ascending=False)
    
    return perm_df

    # Higher value = more important feature, relatively compared to other features
    # Near-zero value = unimportant feature, relatively compared to other features
    # Negative value = noisy feature

In [ ]:
def create_shap_values_plot(model, X, X_train, X_test):
    import shap
    X_train_df = pd.DataFrame(X_train, columns=X.columns.tolist())
    X_test_df = pd.DataFrame(X_test, columns=X.columns.tolist())
    explainer = shap.KernelExplainer(model.predict, X_train_df.sample(200))
    shap_values = explainer.shap_values(X_test_df.sample(100))
    shap.summary_plot(shap_values, X_test_df.sample(100))

In [ ]:
def ols_model(X_true, y_true):
    import statsmodels.api as sm
    X_true_sm = sm.add_constant(X_true)

    # Fit OLS model
    ols_model = sm.OLS(y_true, X_true_sm).fit()

    # Summary table
    print(ols_model.summary())

## Experiments

In [ ]:
X.columns

In [ ]:
# Train and test model
# Train all

selected_vars = [???]
X = orig_X[selected_vars]
y = df["cnt"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

# Construct a model
model = LinearRegression()
model.fit(X_train, y_train)

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

In [ ]:
# Question: What are size of your training dataset vs. testing dataset
# Answer:

In [ ]:
# Train and test model
# Remove variables that are high-correlated with y (so the model does not perfectly predict responses.)

In [ ]:
# Question: What are variables that are high-correlated with y?
# Answer:

In [ ]:
selected_vars = [???]
X = orig_X[selected_vars]
y = df["cnt"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

# Construct a model
model = LinearRegression()
model.fit(X_train, y_train)

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

In [ ]:
coeff_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_
})

print(coeff_df)

In [ ]:
# Train and test model
# Remove multi-collinearity variables (if any)

In [ ]:
# Question: What are pairs or groups of multi-collinearity variables variables among predictors (X)?
# Answer:

In [ ]:
selected_vars = [???]
X = orig_X[selected_vars]
y = df["cnt"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

# Construct a model
model = LinearRegression()
model.fit(X_train, y_train)

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

In [ ]:
# Train and test model
# Continue with transform skewed variables

In [ ]:
# Question: What are skewed variables among predictors (X)?
# Answer:

In [ ]:
# Instruction: Place your code for transforming variables? 

In [ ]:
selected_vars = [???]
X = orig_X[selected_vars]
y = df["cnt"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

# Construct a model
model = LinearRegression()
model.fit(X_train, y_train)

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

In [ ]:
# Question: Does variable transformation help improve regression performance?
# Answer:

In [ ]:
coeff_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_
})

print(coeff_df)

In [ ]:
# Question: Which 3 variables do you think have the MOST effect on the response?
# Answer:

In [ ]:
# Question: Which 3 variables do you think have the LEAST effect on the response?
# Answer:

In [ ]:
# Train and test model
# Use the same set of variables as the ones with transformation.  Now, try scaling

In [ ]:
# Instruction: Place the code for transformation and scaling below.

In [ ]:
selected_vars = [???]
X = orig_X[selected_vars]
y = df["cnt"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

# Construct a model
model = LinearRegression()
model.fit(X_train, y_train)

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

In [ ]:
coeff_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})

print(coeff_df)

In [ ]:
# Question: Does scaling help improve regression performance?
# Answer:

In [ ]:
# Instruction: Place code to create residuals vs fitted plot to see patterns of how residuals spread.

In [ ]:
# Question: What can you summarize about residuals homoscedasticity?
# Answer:

In [ ]:
# Instruction: Place code to create residual qqplot to see normality of residuals.

In [ ]:
# Question: What can you summarize about residuals normality?
# Answer:

In [ ]:
# Check extreme values and create a plot for cooks distance for the first time

In [ ]:
leverage, leverage_threshold, standardized_residuals, cooks_distance, cooks_threshold, index_to_remove = check_extreme_values(X_train, y_train, y_train_pred)
main_index_to_remove = index_to_remove
print(len(main_index_to_remove)/df.shape[0])
create_cooks_plot(X_train, cooks_distance, cooks_threshold)

In [ ]:
# Train and test model
# Use the same set of variables as the ones with transformation and scaling. 
# If transformation or scaling helps with performance, include them in your codebox below

# Try to remove rows with extreme predictor values from the first time

In [ ]:
# Remove extreme #1
# This given code is to remove rows with extreme values from the first time before training and testing model
df = orig_df.copy()
df = df.drop(main_index_to_remove,axis=0)
orig_X = df.drop("cnt", axis=1)
X = orig_X.copy()

selected_vars = [???]
X = orig_X[selected_vars]
y = df["cnt"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

# Construct a model
model = LinearRegression()
model.fit(X_train, y_train)

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

In [ ]:
# Check extreme values and create a plot for cooks distance for the second time

In [ ]:
leverage, leverage_threshold, standardized_residuals, cooks_distance, cooks_threshold, index_to_remove = check_extreme_values(X_train, y_train, y_train_pred)
past_main_index_to_remove = main_index_to_remove
main_index_to_remove = main_index_to_remove + index_to_remove
print(len(main_index_to_remove)/df.shape[0])
create_cooks_plot(X_train, cooks_distance, cooks_threshold)

In [ ]:
# Train and test model
# Use the same set of variables as the ones with transformation and scaling. 
# If transformation or scaling helps with performance, include them in your codebox below

# Try to remove rows with extreme predictor values from the second time

# Place the given code to remove rows with extreme values before training and testing model
'''
df = orig_df.copy()
df = df.drop(main_index_to_remove,axis=0)
orig_X = df.drop("cnt", axis=1)
X = orig_X.copy()
'''

# Note if the new performance drops lower than the previous one, you can use the following code to keep your previous verson of dataset,
'''
df = orig_df.copy()
df = df.drop(past_main_index_to_remove,axis=0)
orig_X = df.drop("cnt", axis=1)
X = orig_X.copy()
'''

In [ ]:
# Remove extreme #2
# This given code is to remove rows with extreme values from the second time before training and testing model
df = orig_df.copy()
df = df.drop(main_index_to_remove,axis=0)
orig_X = df.drop("cnt", axis=1)
X = orig_X.copy()

selected_vars = [???]
X = orig_X[selected_vars]
y = df["cnt"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

# Construct a model
model = LinearRegression()
model.fit(X_train, y_train)

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

In [ ]:
# Check extreme values and create a plot for cooks distance for the second time

In [ ]:
# You can try to remove extreme values until performance does not improve much or the removal percentage is not too much.

In [ ]:
# Optional: 
# Remove extreme #3
# Check extreme values and create a plot for cooks distance for the third time

In [ ]:
# ...

In [ ]:
# Question: When you are done with extreme value removal, what is size of your training dataset now?
# Answer:

In [ ]:
# Place code to compute permutation importance

In [ ]:
# Place code to create plot for shap values 

In [ ]:
# This is to report p-values and confidence of each coefficients
ols_model(X_train, y_train)

In [ ]:
# Question: From permutation importance, shap values, p-values, which one do you think they are "important" varaibles? 
#           Note that the number of important variables should be fewer than original number of varaibles.
# Answer:

In [ ]:
X.columns

In [ ]:
# Train and test model
# Use important variables from your answer above to construct a model 

In [ ]:
# Train and test model

selected_vars = [???]
X = orig_X[selected_vars]
y = df["cnt"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

# Construct a model
model = LinearRegression()
model.fit(X_train, y_train)

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

In [ ]:
# Train and test model
# From your important variables above, reduce one of them.  Use the remaining to construct a model.

In [ ]:
selected_vars = [???]
X = orig_X[selected_vars]
y = df["cnt"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

# Construct a model
model = LinearRegression()
model.fit(X_train, y_train)

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

In [ ]:
# Place code to create residuals vs fitted plot and residual qqplot for the latest regression model 

In [ ]:
# You can use the following code to plot one important variable vs. response and see their relationship.
selected_var = ???
predict_var = 'cnt'
sns.lmplot(x=selected_var, y=predict_var, data=df)